# Kiểm tra 4 file logits/labels (val, test)

Cách dùng: bấm **Runtime → Run all**. Ở ô thứ 2, bấm *Choose files* và chọn **cùng lúc** 4 file:
`logits_val.npy`, `labels_val.npy`, `logits_test.npy`, `labels_test.npy`.
(Nếu có thể, chọn thêm `val.csv` và `test.csv` để kiểm tra thứ tự hàng khớp với CSV.)


In [ ]:
from google.colab import files
up = files.upload()
print(list(up.keys()))


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

COLS = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
        'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
        'Fibrosis', 'Pleural_Thickening', 'Hernia']

for s, n in [('val', 10664), ('test', 25596)]:
    X = np.load(f'logits_{s}.npy')
    Y = np.load(f'labels_{s}.npy')
    print(f'--- {s} ---  logits {X.shape} {X.dtype} | labels {Y.shape} {Y.dtype}')
    assert X.shape == Y.shape == (n, 14), 'SAI KÍCH THƯỚC'
    assert np.isfinite(X).all(), 'Có NaN/inf trong logits'
    assert set(np.unique(Y)) <= {0, 1}, 'Nhãn không phải 0/1'
    print('logits min/max:', round(float(X.min()), 3), round(float(X.max()), 3))
    if X.min() >= 0 and X.max() <= 1:
        print('CẢNH BÁO: logits nằm trong [0, 1], có thể đã bị qua sigmoid')

    aucs = {}
    for j, c in enumerate(COLS):
        if 0 < Y[:, j].sum() < n:
            aucs[c] = roc_auc_score(Y[:, j], X[:, j])
    print(pd.Series(aucs).round(3).to_string())
    print('mean AUROC:', round(float(np.mean(list(aucs.values()))), 4))

    csv = f'{s}.csv'
    if os.path.exists(csv):
        df = pd.read_csv(csv)
        assert len(df) == n, f'{csv} có {len(df)} dòng, cần {n}'
        assert np.array_equal(Y.astype(int), df[COLS].values.astype(int)), 'Nhãn LỆCH so với CSV (thứ tự hàng sai)'
        print('Khớp thứ tự hàng với', csv)
    else:
        print(f'(Chưa upload {csv} nên bỏ qua bước so khớp thứ tự)')

print('XONG: tất cả kiểm tra đã đạt')
